In [1]:
import jax
from jax import numpy as jnp

In [2]:
def calibration_error(logits, labels, num_bins):
    # 1. Wahrscheinlichkeiten und Vorhersagen über die letzte Achse (n_classes)
    probs = jax.nn.softmax(logits, axis=-1)
    confidences = jnp.max(probs, axis=-1)
    predictions = jnp.argmax(probs, axis=-1)
    accuracies = (predictions == labels).astype(jnp.float32)

    # 2. Bins aufbauen
    bin_boundaries = jnp.linspace(0, 1, num_bins + 1)
    bin_lowers = bin_boundaries[:-1]
    bin_uppers = bin_boundaries[1:]

    # Dynamisch die Achse für die Datenpunkte bestimmen (immer die letzte verbleibende)
    # Bei (samples, data) ist es axis=-1. Bei (data,) ist es ebenfalls axis=-1.
    reduce_axis = -1

    def body_fun(ece, bins):
        bin_lower, bin_upper = bins
        
        # Maske hat die Form (samples, data) oder (data,)
        in_bin = jnp.logical_and(confidences > bin_lower, confidences <= bin_upper)
        in_bin_float = in_bin.astype(jnp.float32)

        # Anzahl der Elemente in diesem Bin entlang der Daten-Achse
        bin_count = jnp.sum(in_bin_float, axis=reduce_axis)
        # Gesamtzahl der Datenpunkte entlang dieser Achse
        total_count = in_bin_float.shape[reduce_axis]
        
        # Anteil der Datenpunkte in diesem Bin (pro Sample)
        prop_in_bin = bin_count / total_count
        
        # JIT-freundliche Berechnung: Teilen durch 1.0 falls bin_count == 0
        safe_bin_count = jnp.where(bin_count > 0, bin_count, 1.0)
        
        accuracy_in_bin = jnp.sum(jnp.where(in_bin, accuracies, 0.0), axis=reduce_axis) / safe_bin_count
        avg_confidence_in_bin = jnp.sum(jnp.where(in_bin, confidences, 0.0), axis=reduce_axis) / safe_bin_count
        
        # Wenn bin_count == 0 war, setzen wir die Differenz auf 0 (da safe_bin_count ungleich 0 war)
        bin_ece = jnp.abs(avg_confidence_in_bin - accuracy_in_bin) * prop_in_bin
        bin_ece = jnp.where(bin_count > 0, bin_ece, 0.0)
        
        ece += bin_ece
        return ece, None

    # ece startet als Nullen-Array mit der Form der verbleibenden Batch-Dimensionen
    # Bei (samples, data) -> ece Form: (samples,)
    # Bei (data,)          -> ece Form: () (Skalar)
    init_ece = jnp.zeros(logits.shape[:-1])
    
    ece, _ = jax.lax.scan(body_fun, init_ece, (bin_lowers, bin_uppers))
    return ece

In [8]:
logits = jax.random.normal(jax.random.PRNGKey(5), (5, 10, 3))  # (n_samples, n_data, n_classes)
labels = jnp.array([0, 1, 2, 0, 1, 2, 0, 1, 2, 0])  # (n_data,)
ece = calibration_error(logits[0], labels, num_bins=15)
print("ECE:", ece)

ECE: [0.2818678 0.2818678 0.2818678 0.2818678 0.2818678 0.2818678 0.2818678
 0.2818678 0.2818678 0.2818678]


In [13]:
def calibration_error(logits, labels, num_bins):
    probs = jax.nn.softmax(logits, axis=-1)
    confidences = jnp.max(probs, axis=-1)
    predictions = jnp.argmax(probs, axis=-1)
    accuracies = (predictions == labels).astype(jnp.float32)

    bin_boundaries = jnp.linspace(0, 1, num_bins + 1)
    bin_lowers = bin_boundaries[:-1]
    bin_uppers = bin_boundaries[1:]

    def body_fun(ece, bins):
        bin_lower, bin_upper = bins
        # get mask for samples with confidence in (bin_lower, bin_upper]
        in_bin = jnp.logical_and(confidences > bin_lower, confidences <= bin_upper)

        prop_in_bin = jnp.mean(in_bin.astype(jnp.float32))
        # same as jnp.mean(accuracies[in_bin]) but jit friendly
        accuracy_in_bin = jnp.where(
            prop_in_bin > 0,
            jnp.sum(jnp.where(in_bin, accuracies, 0.0)) / jnp.sum(in_bin),
            0.0,
        )
        # same as jnp.mean(confidences[in_bin]) but jit friendly
        avg_confidence_in_bin = jnp.where(
            prop_in_bin > 0,
            jnp.sum(jnp.where(in_bin, confidences, 0.0)) / jnp.sum(in_bin),
            0.0,
        )
        ece += jnp.abs(avg_confidence_in_bin - accuracy_in_bin) * prop_in_bin
        return ece, None

    ece, _ = jax.lax.scan(body_fun, 0.0, (bin_lowers, bin_uppers))
    return ece

In [12]:
ece = calibration_error(logits[0], labels, num_bins=15)
print("ECE:", ece)

ECE: 0.2818678


In [17]:
jax.vmap(calibration_error, in_axes=(0, None, None))(logits, labels, 15)

Array([0.2818678 , 0.5327313 , 0.3912438 , 0.43886486, 0.5163623 ],      dtype=float32)

In [21]:

post_predict_probs = jax.nn.softmax(logits[0])
true_probs = jax.nn.one_hot(labels, logits.shape[-1])
brier = jnp.mean(jnp.sum((post_predict_probs - true_probs) ** 2, axis=-1))
brier

Array(0.5652626, dtype=float32)

In [20]:

post_predict_probs = jax.nn.softmax(logits)
true_probs = jax.nn.one_hot(labels, logits.shape[-1])
brier = jnp.mean(jnp.sum((post_predict_probs - true_probs) ** 2, axis=-1), axis=-1)
brier

Array([0.5652626 , 1.0369847 , 0.71812767, 0.969783  , 0.9453204 ],      dtype=float32)